In [ ]:
# uncomment if you wish to run this notebook on google colab with dataset on google drive

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
dataset_path = "path/to/dataset/folder" # fill with path to where you downloaded the dataset
allSubjects = [f"Subject_{i+1:02d}" for i in range(20)]
N_EMG_CHANNEL = 8
N_KINEMATIC_CHANNEL = 15
EMG_S_FREQ = 500

## General imports and setup

In [ ]:
!pip install pyriemann
!pip install mne

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyriemann.tangentspace import TangentSpace
from pyriemann.estimation import Covariances
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import mne
from keras.src.callbacks import EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K
import gc
from copy import deepcopy
from scipy.ndimage import gaussian_filter1d

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

In [ ]:
!nvidia-smi
tf.config.list_physical_devices('GPU')

# Code

In [ ]:
def loadData(sName):
  data = mne.io.read_raw_fif(f"{dataset_path}/{sName}.fif", preload=True)

  emg = data.get_data(picks=[f"EMG {i+1}" for i in range(N_EMG_CHANNEL)]).T
  label = data.get_data(picks=[f"Angle {i+1}" for i in range(N_KINEMATIC_CHANNEL)]).T

  emg = mne.filter.filter_data(emg.T, EMG_S_FREQ, 15, 150, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 50, notch_widths=5, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 100, notch_widths=5, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 150, notch_widths=5, verbose="CRITICAL").T

  return emg, label

In [ ]:
def sync(emg, label):
  sfreq = EMG_S_FREQ
  ch_names = [f"emg{i+1}" for i in range(N_EMG_CHANNEL)]
  info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="emg")
  raw = mne.io.RawArray(deepcopy(emg).T, info)
  raw.filter(15, 50, picks=ch_names)

  raw_envelope = raw.apply_hilbert(envelope=True, picks=ch_names)
  envelope = raw_envelope.get_data()

  # decomposition into move and hold commands
  S = envelope
  M = np.zeros(S.shape)
  H = np.zeros(S.shape)

  for i in range(1, S.shape[1]):
      M[:, i] = S[:, i] - H[:, i-1]
      H[:, i] = H[:, i-1] + M[:, i]/sfreq

  l = np.apply_along_axis(gaussian_filter1d, 0, label, sigma=20)
  speed = np.gradient(l)[0]
  speed = np.apply_along_axis(gaussian_filter1d, 0, speed, sigma=20)

  M2 = np.apply_along_axis(gaussian_filter1d, 0, M.T, sigma=20).T

  CUT = 4
  shiftPerSegment = []

  for i in range(CUT):
    idx = np.arange(len(speed)*i//CUT, len(speed)*(i+1)//CUT)

    M2_ = M2[:, idx]
    speed_ = speed[idx]

    shift = 0
    cor = []
    l = len(speed_)
    for shift in range(-200, 501, 5):
      if shift >=0:
        a = np.max(M2_[:, :l-shift]**2, axis=0)**0.5
        b = np.max(speed_[shift:]**2, axis=1)**0.5
      else:
        a = np.max(M2_[:, -shift:]**2, axis=0)**0.5
        b = np.max(speed_[:l+shift]**2, axis=1)**0.5

      cor.append((shift, np.corrcoef(a, b)[0, 1]))
    cor = np.array(cor)

    m = cor[np.argmax(cor[:, 1]), 0]
    shiftPerSegment.append(m)

  label_sync = np.concatenate([label[np.array(np.clip(np.arange(len(speed)*i//CUT, len(speed)*(i+1)//CUT) + shiftPerSegment[i], 0, len(label)-1), dtype=int)] for i in range(CUT)])
  return label_sync

In [ ]:
def extractSamples_10fps(emg, label):
    step = 50
    size = 150
    ts = np.arange(2000, len(emg)-2000, step)

    x_emg = []
    y = []
    for t in ts:
        x_emg.append(emg[t - size+1:t+1].transpose())
        y.append(label[t])
    x_emg = np.array(x_emg)
    y = np.array(y)

    return x_emg, y

In [ ]:
def createSequences(x, y):
  sequenceLength = 10
  sequence = []
  Y = []
  for i in range(sequenceLength, len(x)-1):
      sequence.append(x[i-sequenceLength+1:i+1])
      Y.append(y[i])
  sequence = np.array(sequence)
  y = np.array(Y)
  return sequence, y

In [ ]:
def getsync_data(subjectName, multipleFreqBands=False):
  emg, label = loadData(subjectName)
  label_sync = sync(emg, label)

  if multipleFreqBands:
    x = []
    sfreq = EMG_S_FREQ
    ch_names = [f"emg{i+1}" for i in range(N_EMG_CHANNEL)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="emg")
    bands = [(15, 40), (40, 80), (80, 150)]
    for lb, hb in bands:
      emg_band = mne.io.RawArray(deepcopy(emg.T), info)
      emg_band.filter(lb, hb, picks=ch_names)
      emg_band = emg_band.get_data().T
      x_band, y = extractSamples_10fps(emg_band, label_sync)
      x.append(x_band)
    x = np.array(x).transpose((1, 0, 2, 3))
    return x, y

  else:
    x, y = extractSamples_10fps(emg, label_sync)
    return x, y

In [ ]:
# complete model
def covMatTangentSpace_3FreqBands(xTrain, xTest):
  """
  needs to have extracted samples in multiple frequency bands
  """
  xTrainBands = []
  xTestBands = []
  for i in range(3):
    cmtsExtractor = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace('riemann'))])
    cmtsExtractor.fit(xTrain[:, i])
    cstm_train = cmtsExtractor.transform(xTrain[:, i])
    cstm_test = cmtsExtractor.transform(xTest[:, i])
    xTrainBands.append(cstm_train)
    xTestBands.append(cstm_test)
  cstm_train = np.concatenate(xTrainBands, axis=1)
  cstm_test = np.concatenate(xTestBands, axis=1)
  return cstm_train, cstm_test


# simplified model
def covMatTangentSpace(xTrain, xTest):
  cmtsExtractor = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace('riemann'))])
  cmtsExtractor.fit(xTrain)
  cstm_train = cmtsExtractor.transform(xTrain)
  cstm_test = cmtsExtractor.transform(xTest)
  return cstm_train, cstm_test

In [ ]:
def createRNN(output_dim=15, input_shape=None):
    model = keras.Sequential(name="Fast_GRU_Model")
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Dense(256, activation="tanh"))

    model.add(layers.GRU(256, return_sequences=True, dropout=0.1, activation="tanh"))
    model.add(layers.GRU(128, return_sequences=False, dropout=0.1, activation="tanh"))

    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(output_dim, activation="linear", dtype="float32"))  # ensure float32 output

    optimizer = Adam(learning_rate=2e-4, clipnorm=1.0)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)
    model.compile(optimizer=optimizer, loss="huber", metrics=["mae"])
    return model

tf.config.optimizer.set_jit(True)


def make_dataset(X, Y, batchSize=256):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    return ds.shuffle(len(X)).batch(batchSize).prefetch(tf.data.AUTOTUNE)


def rnn(xTrain, yTrain, xTest):
    batchSize = 256

    trainIdx = np.arange(len(xTrain))
    valSize = len(xTrain)//10 # use 10% of training data for validation to find convergence
    valIdx_start = np.random.randint(len(xTrain) - valSize)
    valIdx = trainIdx[valIdx_start:valIdx_start+valSize]
    trainIdx = np.delete(trainIdx, np.arange(valIdx_start, valIdx_start+valSize))
    np.random.shuffle(trainIdx)

    # Data subsets
    trainX, trainY = xTrain[trainIdx], yTrain[trainIdx]
    valX, valY = xTrain[valIdx], yTrain[valIdx]

    # Model
    reg = createRNN(output_dim=trainY.shape[-1], input_shape=(xTrain.shape[1], xTrain.shape[2]))

    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    lr_schedule = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    train_ds = make_dataset(trainX, trainY, batchSize)
    val_ds = make_dataset(valX, valY, batchSize)

    history = reg.fit(
        train_ds,
        validation_data=val_ds,
        epochs=200,
        callbacks=[early_stop, lr_schedule],
        verbose=0
    )
    yPred = reg.predict(xTest, batch_size=512)


    plt.rcParams['figure.figsize'] = [5, 5]
    plt.plot(history.history['loss'], label="loss")
    plt.plot(history.history['val_loss'], label="val loss")
    plt.show()

    # cleanup
    K.clear_session()
    del reg
    gc.collect()

    return yPred

# 1. complete TRR model

In [ ]:
x, y = getsync_data(allSubjects[0], multipleFreqBands=True)

In [ ]:
minutesOfTrainingData = 27
sequenceLength = 10

prediction = []
groundTruth = []
for foldId, (trainIdx, testIdx) in enumerate(KFold(10, shuffle=False).split(x)):
  print(f"-- Fold {foldId+1}/10 --")

  trainDataSize = minutesOfTrainingData*600 # 10 sample per second
  startTrainIdx = np.random.randint(max(1, len(trainIdx) - trainDataSize))
  trainIdx = trainIdx[startTrainIdx:startTrainIdx+trainDataSize]

  xTrain, xTest = deepcopy(x[trainIdx]), deepcopy(x[testIdx])
  yTrain, yTest = deepcopy(y[trainIdx]), deepcopy(y[testIdx])

  for c in range(N_EMG_CHANNEL):
    for band in range(xTrain.shape[1]): # standardization when we have multiple frequency bands of signal
      m = np.mean(xTrain[:, band, c, :])
      s = np.std(xTrain[:, band, c, :])
      xTrain[:, band, c, :] = (xTrain[:, band, c, :] - m) / s
      xTest[:, band, c, :] = (xTest[:, band, c, :] - m) / s

  xTrain, xTest = covMatTangentSpace_3FreqBands(xTrain, xTest)

  xTrain, yTrain = createSequences(xTrain, yTrain)
  xTest, yTest = createSequences(xTest, yTest)

  labelScaler = StandardScaler().fit(yTrain)
  yTrain_scaled = labelScaler.transform(yTrain)

  yPred = rnn(xTrain, yTrain_scaled, xTest)

  yPred_unscaled = labelScaler.inverse_transform(yPred)

  prediction.extend(yPred_unscaled)
  groundTruth.extend(yTest)

prediction = np.array(prediction)
groundTruth = np.array(groundTruth)

In [ ]:
plt.rcParams['figure.figsize'] = [20, 7]
maxIdx = 2000
plt.plot(groundTruth[:maxIdx] + np.arange(15)*200, color="black")
plt.plot(prediction[:maxIdx] + np.arange(15)*200, color="tab:blue")
for i in range(15):
  plt.fill_between(np.arange(maxIdx), groundTruth[:maxIdx, i] + i*200, prediction[:maxIdx, i] + i*200, color="tab:red", alpha=0.5)
plt.show()

plt.rcParams['figure.figsize'] = [20, 5]
plt.title("Distributions of real angles (blue) and predicted angles (orange)")
plt.violinplot(groundTruth, side="low")
plt.violinplot(prediction, side="high")
plt.show()

plt.rcParams['figure.figsize'] = [50, 3]
fig, ax = plt.subplots(1, 15)
for angle in range(15):
  ax[angle].scatter(groundTruth[:, angle], prediction[:, angle], s=1, alpha=0.1)
  ax[angle].plot([min(groundTruth[:, angle]), max(groundTruth[:, angle])], [min(groundTruth[:, angle]), max(groundTruth[:, angle])], color="tab:red", linestyle="--")
  ax[angle].set_title(f"Angle {angle+1}")
  ax[angle].set_xlabel("Ground truth")
  ax[angle].set_ylabel("Prediction")
plt.show()

# 2. simplified TRR model

In [ ]:
x, y = getsync_data(allSubjects[0], multipleFreqBands=False)

In [ ]:
minutesOfTrainingData = 27
sequenceLength = 10

prediction = []
groundTruth = []
for foldId, (trainIdx, testIdx) in enumerate(KFold(10, shuffle=False).split(x)):
  print(f"-- Fold {foldId+1}/10 --")

  trainDataSize = minutesOfTrainingData*600 # 10 sample per second
  startTrainIdx = np.random.randint(max(1, len(trainIdx) - trainDataSize))
  trainIdx = trainIdx[startTrainIdx:startTrainIdx+trainDataSize]

  xTrain, xTest = deepcopy(x[trainIdx]), deepcopy(x[testIdx])
  yTrain, yTest = deepcopy(y[trainIdx]), deepcopy(y[testIdx])

  for c in range(N_EMG_CHANNEL):
    m = np.mean(xTrain[:, c, :])
    s = np.std(xTrain[:, c, :])
    xTrain[:, c, :] = (xTrain[:, c, :] - m) / s
    xTest[:, c, :] = (xTest[:, c, :] - m) / s

  xTrain, xTest = covMatTangentSpace(xTrain, xTest)

  xTrain, yTrain = createSequences(xTrain, yTrain)
  xTest, yTest = createSequences(xTest, yTest)

  labelScaler = StandardScaler().fit(yTrain)
  yTrain_scaled = labelScaler.transform(yTrain)

  yPred = rnn(xTrain, yTrain_scaled, xTest)

  yPred_unscaled = labelScaler.inverse_transform(yPred)

  prediction.extend(yPred_unscaled)
  groundTruth.extend(yTest)

prediction = np.array(prediction)
groundTruth = np.array(groundTruth)

In [ ]:
plt.rcParams['figure.figsize'] = [20, 7]
maxIdx = 2000
plt.plot(groundTruth[:maxIdx] + np.arange(15)*200, color="black")
plt.plot(prediction[:maxIdx] + np.arange(15)*200, color="tab:blue")
for i in range(15):
  plt.fill_between(np.arange(maxIdx), groundTruth[:maxIdx, i] + i*200, prediction[:maxIdx, i] + i*200, color="tab:red", alpha=0.5)
plt.show()

plt.rcParams['figure.figsize'] = [20, 5]
plt.title("Distributions of real angles (blue) and predicted angles (orange)")
plt.violinplot(groundTruth, side="low")
plt.violinplot(prediction, side="high")
plt.show()

plt.rcParams['figure.figsize'] = [50, 3]
fig, ax = plt.subplots(1, 15)
for angle in range(15):
  ax[angle].scatter(groundTruth[:, angle], prediction[:, angle], s=1, alpha=0.1)
  ax[angle].plot([min(groundTruth[:, angle]), max(groundTruth[:, angle])], [min(groundTruth[:, angle]), max(groundTruth[:, angle])], color="tab:red", linestyle="--")
  ax[angle].set_title(f"Angle {angle+1}")
  ax[angle].set_xlabel("Ground truth")
  ax[angle].set_ylabel("Prediction")
plt.show()